# RC0 fresh8 method-stability base generation

DIAGNOSTIC_ONLY / METHOD_ONLY. Generates exactly eight carrier-free base MP4s. Phase C is not executed here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, zipfile

REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git'
AUTHORIZED_REF = 'ba226dc6e3d11f501c9fded970d2b8c68d3c6546'
RUN_ID = 'f37e12c0e6a325fe'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/SC-SSTW-Feasibility/rc0-fresh8-method-stability'
AUTHORIZE_EXECUTION = True
AUTHORIZE_DRIVE_IO = True
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
MODEL_REVISION = '0fad780a534b6463e45facd96134c9f345acfa5b'


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require_absent(*paths):
    for path in paths:
        if path.exists() or path.is_symlink():
            raise RuntimeError(f'refusing to overwrite: {path}')

if not AUTHORIZE_EXECUTION or not AUTHORIZE_DRIVE_IO:
    raise RuntimeError('explicit user execution and Drive acknowledgements are required')
if not re.fullmatch(r'[0-9a-f]{40}', AUTHORIZED_REF) or not re.fullmatch(r'[0-9a-f]{16}', RUN_ID):
    raise RuntimeError('exact ref and frozen run id are required')
WORK = Path('/content') / f'sc-sstw-fresh8-{RUN_ID}'
OUTPUT = Path('/content') / f'rc0-fresh8-output-{RUN_ID}'
LOG = Path('/content') / f'rc0-fresh8-log-{RUN_ID}'
BUNDLE = Path('/content') / f'rc0-fresh8-bundle-{RUN_ID}'
ARCHIVE = Path('/content') / f'rc0-fresh8-{RUN_ID}.zip'
SIDECAR = Path('/content') / f'rc0-fresh8-{RUN_ID}.zip.sha256.json'
DRIVE_ROOT = Path(DRIVE_OUTPUT_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ARCHIVE = DRIVE_ROOT / ARCHIVE.name
DRIVE_SIDECAR = DRIVE_ROOT / SIDECAR.name
require_absent(WORK, OUTPUT, LOG, BUNDLE, ARCHIVE, SIDECAR, DRIVE_ARCHIVE, DRIVE_SIDECAR)
LOG.mkdir()
runner_started = False
completed = None
caught = None
try:
    with (LOG/'clone.stdout').open('xb') as out, (LOG/'clone.stderr').open('xb') as err:
        subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(WORK)], check=True, stdout=out, stderr=err)
    with (LOG/'checkout.stdout').open('xb') as out, (LOG/'checkout.stderr').open('xb') as err:
        subprocess.run(['git', 'checkout', '--detach', AUTHORIZED_REF], cwd=WORK, check=True, stdout=out, stderr=err)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=WORK, check=True, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(['git', 'status', '--porcelain=v1', '--untracked-files=all'], cwd=WORK, check=True, capture_output=True, text=True).stdout
    if head != AUTHORIZED_REF or dirty:
        raise RuntimeError('clean exact-ref checkout failed')
    config_path = WORK/'configs/rc0_fresh8_method_stability_gpu.json'
    config_sha = sha256_file(config_path)
    expected_run_id = hashlib.sha256(('RC0-FRESH8-METHOD-STABILITY:' + AUTHORIZED_REF + ':' + config_sha).encode()).hexdigest()[:16]
    if RUN_ID != expected_run_id:
        raise RuntimeError('prefilled run id does not bind exact ref and config')
    locked = ['accelerate==1.4.0', 'diffusers==0.35.2', 'ftfy==6.3.1', 'huggingface_hub==0.35.3', 'imageio==2.37.0', 'imageio_ffmpeg==0.6.0', 'numpy==1.26.4', 'safetensors==0.5.3', 'transformers==4.49.0']
    with (LOG/'pip.stdout').open('xb') as out, (LOG/'pip.stderr').open('xb') as err:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *locked], check=True, stdout=out, stderr=err)
    import torch
    from huggingface_hub import snapshot_download
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('CUDA BF16 runtime unavailable')
    snapshot = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, local_files_only=False))
    resolved_snapshot = snapshot.resolve(strict=True)
    if snapshot.is_symlink() or not snapshot.is_dir() or snapshot.name != MODEL_REVISION or resolved_snapshot.name != MODEL_REVISION:
        raise RuntimeError('downloaded snapshot does not resolve to the frozen revision')
    runtime = {'head': head, 'config_sha256': config_sha, 'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'snapshot': str(resolved_snapshot), 'diagnostic_class': 'DIAGNOSTIC_ONLY'}
    (LOG/'runtime.json').write_text(json.dumps(runtime, sort_keys=True, indent=2)+'\n')
    argv = [sys.executable, str(WORK/'experiments/run_rc0_fresh8_method_stability_gpu.py'), '--output', str(OUTPUT)]
    runner_started = True
    with (LOG/'runner.stdout').open('xb') as out, (LOG/'runner.stderr').open('xb') as err:
        completed = subprocess.run(argv, cwd=WORK, stdout=out, stderr=err)
    if completed.returncode != 0:
        raise RuntimeError(f'base runner returned {completed.returncode}')
except BaseException as exc:
    caught = exc
finally:
    BUNDLE.mkdir()
    shutil.copytree(LOG, BUNDLE/'log')
    if OUTPUT.exists() and OUTPUT.is_dir() and not OUTPUT.is_symlink():
        shutil.copytree(OUTPUT, BUNDLE/'output')
    state = {'schema': 'sc-sstw.rc0.fresh8-notebook-state.v1', 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'authorized_ref': AUTHORIZED_REF, 'runner_started': runner_started, 'runner_return_code': None if completed is None else completed.returncode, 'failure_type': None if caught is None else type(caught).__name__, 'formal_result': False, 'stage_progression_allowed': False}
    (BUNDLE/'notebook_state.json').write_text(json.dumps(state, sort_keys=True, indent=2)+'\n')
    shutil.make_archive(str(ARCHIVE.with_suffix('')), 'zip', BUNDLE)
    with zipfile.ZipFile(ARCHIVE) as handle:
        if handle.testzip() is not None:
            raise RuntimeError('local ZIP integrity failed')
    archive_sha = sha256_file(ARCHIVE)
    sidecar = {'schema': 'sc-sstw.rc0.fresh8-sidecar.v1', 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'source_ref': AUTHORIZED_REF, 'archive_name': ARCHIVE.name, 'archive_size': ARCHIVE.stat().st_size, 'archive_sha256': archive_sha, 'formal_result': False, 'stage_progression_allowed': False}
    SIDECAR.write_text(json.dumps(sidecar, sort_keys=True, indent=2)+'\n')
    with ARCHIVE.open('rb') as source, DRIVE_ARCHIVE.open('xb') as target:
        shutil.copyfileobj(source, target)
    with SIDECAR.open('rb') as source, DRIVE_SIDECAR.open('xb') as target:
        shutil.copyfileobj(source, target)
    if DRIVE_ARCHIVE.stat().st_size != ARCHIVE.stat().st_size or sha256_file(DRIVE_ARCHIVE) != archive_sha:
        raise RuntimeError('Drive ZIP readback mismatch')
    if DRIVE_SIDECAR.read_bytes() != SIDECAR.read_bytes():
        raise RuntimeError('Drive sidecar readback mismatch')
    with zipfile.ZipFile(DRIVE_ARCHIVE) as handle:
        if handle.testzip() is not None:
            raise RuntimeError('Drive ZIP integrity failed')
if caught is not None:
    raise caught
print({'status': 'BASE_EXACT8_NOTEBOOK_PACKAGED', 'run_id': RUN_ID, 'drive_zip': str(DRIVE_ARCHIVE), 'drive_sidecar': str(DRIVE_SIDECAR)})
